In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [3]:
air_quality = pd.read_csv('../data/processed_air_quality_data.csv')
pm_detail=pd.read_csv('../data/processed_pm_detail.csv')
weather=pd.read_csv('../data/processed_weather_data.csv')

In [6]:
weather.columns

Index(['pressure_hPa', 'datetime', 'precipitation_mm', 'wind_direction_deg',
       'temperature_C', 'relative_humidity_pct', 'visibility_m',
       'wind_speed_ms'],
      dtype='object')

In [25]:
air_quality['datetime'].dtype, weather['datetime'].dtype, pm_detail['datetime'].dtype

(dtype('<M8[ns]'), dtype('<M8[ns]'), dtype('<M8[ns]'))

In [4]:
air_quality['datetime'] = pd.to_datetime(air_quality['datetime'], errors='coerce')
weather['datetime'] = pd.to_datetime(weather['datetime'], errors='coerce')
pm_detail['datetime'] = pd.to_datetime(pm_detail['datetime'], errors='coerce')

In [51]:
air_quality.head()

,O3,PM25,district,CO,NO2,datetime,SO2,PM10
0,47.5,42.0,南山区,0.650,29.5,2020-01-01,4.5,75.5
1,52.5,38.5,坪山区,0.953,20.0,2020-01-01,8.5,71.0
2,68.5,34.0,大鹏新区,0.750,11.5,2020-01-01,5.5,49.5
3,53.0,49.0,宝安区,0.900,41.0,2020-01-01,6.0,88.0
4,57.0,41.0,深圳市,0.700,25.0,2020-01-01,6.0,66.0


In [27]:
weather.head()

,pressure_hPa,datetime,precipitation_mm,wind_direction_deg,temperature_C,relative_humidity_pct,visibility_m,wind_speed_ms
0,1019.0,2020-01-01 00:00:00,0.0,106.0,17.9,82.0,10746.0,0.8
1,1018.9,2020-01-01 01:00:00,0.0,111.0,17.9,81.0,10957.0,1.3
2,1018.5,2020-01-01 02:00:00,0.0,105.0,17.7,82.0,10035.0,0.7
3,1018.3,2020-01-01 03:00:00,0.0,113.0,17.5,82.0,10267.0,0.9
4,1018.2,2020-01-01 04:00:00,0.0,24.0,17.2,85.0,10398.0,0.7


In [39]:
pm_detail.head()

,datetime,site_id,average_pm25_hour,district
0,2020-01-01 03:00:00,通心岭,48.0,福田区
1,2020-01-01 04:00:00,华侨城,50.0,南山区
2,2020-01-01 04:00:00,南海,39.0,南山区
3,2020-01-01 04:00:00,南澳,35.0,大鹏新区
4,2020-01-01 04:00:00,葵涌,35.0,大鹏新区


接下来处理遥测再分析数据。由于本段撰写时没有下载完完整数据集，所以以2020年-2021年数据作为演示

研究区域覆盖深圳陆地及近海水域，海陆下垫面差异可能影响气象变量的区域代表性。为降低海域网格对城市尺度气象特征的影响，本文在空间聚合时剔除位于西南角与东南角、海域占比显著的网格点，对剩余网格进行空间平均以表征深圳陆地区域气象条件。考虑大鹏新区位于研究区东南沿海且地理位置相对独立，采用与其空间邻近的东南角网格作为该区气象代表值。

In [7]:
import pandas as pd
reanalysis_data = pd.read_csv('../data/2020-2021.csv')
reanalysis_data.rename(columns={'time': 'datetime'}, inplace=True)
reanalysis_data["datetime"] = pd.to_datetime(reanalysis_data["datetime"], format="mixed", errors="raise")
reanalysis_data.sort_values(by='datetime', inplace=True)


In [8]:
reanalysis_data.columns

Index(['datetime', 'latitude', 'longitude', 'number', 'expver', 'u10', 'v10',
       't2m_c', 'd2m_c', 'sp_hpa', 'blh_m', 'wind_speed', 'wind_dir', 'rh',
       'source_zip'],
      dtype='object')

In [10]:
reanalysis_data = reanalysis_data.copy()
reanalysis_data["datetime"] = pd.to_datetime(reanalysis_data["datetime"], errors="coerce")
# 找出 8 个网格点坐标，并定位“左下(SW)”和“右下(SE)”
grid = (
    reanalysis_data[["latitude", "longitude"]]
    .drop_duplicates()
    .sort_values(["latitude", "longitude"], ascending=[False, True])
    .reset_index(drop=True)
)

lat_min = grid["latitude"].min()
south_row = grid[np.isclose(grid["latitude"].values, lat_min)]

# 左下：南排最小经度；右下：南排最大经度
sw = south_row.sort_values("longitude").iloc[0]
se = south_row.sort_values("longitude").iloc[-1]

sw_lat, sw_lon = float(sw["latitude"]), float(sw["longitude"])
se_lat, se_lon = float(se["latitude"]), float(se["longitude"])

# 3) 选出要聚合的数值列（排除坐标列；time后面 groupby 用）
numeric_cols = reanalysis_data.select_dtypes(include="number").columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ["latitude", "longitude"]]

# 4) “大鹏区” = 右下(SE)整个网格（该网格下所有变量按 time 聚合）
mask_dp = np.isclose(reanalysis_data["latitude"].values, se_lat) & np.isclose(
    reanalysis_data["longitude"].values, se_lon
)
df_dp = (
    reanalysis_data.loc[mask_dp]
    .groupby("datetime", as_index=False)[numeric_cols]
    .mean()
)
df_dp["district"] = "大鹏区"

# 5) “其它” = 剔除 左下(SW) 和 右下(SE) 后，其余网格按 time 求平均
mask_sw = np.isclose(reanalysis_data["latitude"].values, sw_lat) & np.isclose(
    reanalysis_data["longitude"].values, sw_lon
)
mask_se = mask_dp  # 右下
mask_other = ~(mask_sw | mask_se)

df_other = (
    reanalysis_data.loc[mask_other]
    .groupby("datetime", as_index=False)[numeric_cols]
    .mean()
)
df_other["district"] = "其它"

# 6) 合并成一个 df，并按时间排序
reanalysis_by_district = (
    pd.concat([df_other, df_dp], ignore_index=True)
    .sort_values(["datetime", "district"])
    .reset_index(drop=True)
)

reanalysis_by_district.head()

,datetime,number,expver,u10,v10,t2m_c,d2m_c,sp_hpa,blh_m,wind_speed,wind_dir,rh,district
0,2020-01-01 00:00:00,0.0,1.0,-2.662069,-1.045527,15.878062,12.418854,1020.297550,387.338620,2.869726,67.817326,79.974575,其它
1,2020-01-01 00:00:00,0.0,1.0,-5.173950,-2.068314,16.481903,13.555573,1021.944200,526.619900,5.572045,68.210660,82.841760,大鹏区
2,2020-01-01 01:00:00,0.0,1.0,-3.147481,-1.151866,15.898214,12.456838,1020.544115,459.398925,3.356667,69.468003,80.089157,其它
3,2020-01-01 01:00:00,0.0,1.0,-5.691101,-2.125824,16.141052,13.327606,1022.165800,560.398900,6.075175,69.517580,83.412170,大鹏区
4,2020-01-01 02:00:00,0.0,1.0,-3.309372,-1.326365,17.272919,12.413930,1020.825500,540.437750,3.570108,67.815725,73.181199,其它


In [11]:
reanalysis_by_district.drop(columns=["number", "expver"], inplace=True, errors="ignore")

In [12]:
reanalysis_by_district.tail()

,datetime,u10,v10,t2m_c,d2m_c,sp_hpa,blh_m,wind_speed,wind_dir,rh,district
35083,2021-12-31 21:00:00,-4.359238,-2.381775,15.712677,13.000977,1018.327940,328.935000,4.967474,61.348938,83.918090,大鹏区
35084,2021-12-31 22:00:00,-1.510864,-1.001577,14.211477,12.666616,1016.984480,123.305905,1.863566,54.973054,90.474714,其它
35085,2021-12-31 22:00:00,-3.281860,-2.121368,15.846893,13.092072,1018.472840,281.139220,3.907789,57.121704,83.697470,大鹏区
35086,2021-12-31 23:00:00,-1.350718,-1.150955,14.189158,12.575979,1017.298567,117.397031,1.818981,48.460541,90.071137,其它
35087,2021-12-31 23:00:00,-3.113739,-2.048904,15.860382,12.977997,1018.771850,273.022030,3.727382,56.654236,83.004490,大鹏区


In [13]:
air_quality['datetime'].dtypes

dtype('<M8[ns]')

In [14]:
starttime=reanalysis_by_district["datetime"].min()
endtime=reanalysis_by_district["datetime"].max()
air_quality_data=air_quality[(air_quality["datetime"]>=starttime) & (air_quality["datetime"]<=endtime)]
weather_data=weather[(weather["datetime"]>=starttime) & (weather["datetime"]<=endtime)]
pm_detail_data=pm_detail[(pm_detail["datetime"]>=starttime) & (pm_detail["datetime"]<=endtime)]# 只保留与 reanalysis_by_district 时间范围重叠的数据

weather数据集时间频率处理

In [28]:
def hourly_inhour_rolling(
    weather: pd.DataFrame,
    dt_col: str = "datetime",
    wind_dir_col: str = "wind_direction_deg",
    min_obs: int = 1,              # 每个小时窗口至少需要多少条观测才算有效
    strict_wind_dir: bool = True,  # True: 用 sin/cos 做风向平均；False: 直接算术平均
) -> pd.DataFrame:
    """
    将不规则/混合频率的地面气象数据，按“整点小时内窗口 (t-1h, t]”聚合到小时级。

    - mean: temperature_C, pressure_hPa, relative_humidity_pct, visibility_m, wind_speed_ms
    - sum : precipitation_mm（小时累计）
    - wind_direction_deg: strict_wind_dir=True 时用矢量平均，避免 350°/10° 均值错误
    """
    df = weather.copy()

    # 1) datetime 处理
    df[dt_col] = pd.to_datetime(df[dt_col], errors="coerce")
    df = df.dropna(subset=[dt_col]).sort_values(dt_col)
    df = df.set_index(dt_col)

    # 2) 定义滚动窗口聚合：rolling('60min') 是“向后窗口”，包含 (t-60min, t] 的观测
    mean_cols = ["temperature_C", "pressure_hPa", "relative_humidity_pct", "visibility_m", "wind_speed_ms"]
    sum_cols = ["precipitation_mm"]

    # 确保列存在（缺列就跳过，避免 KeyError）
    mean_cols = [c for c in mean_cols if c in df.columns]
    sum_cols = [c for c in sum_cols if c in df.columns]

    rolling_obj = df.rolling("60min", closed="right")

    out_parts = []

    if mean_cols:
        out_parts.append(rolling_obj[mean_cols].mean())
    if sum_cols:
        out_parts.append(rolling_obj[sum_cols].sum())

    # 3) 风向处理
    if wind_dir_col in df.columns:
        if strict_wind_dir:
            # 角度 -> 弧度
            rad = np.deg2rad(df[wind_dir_col].astype(float))
            sinv = np.sin(rad)
            cosv = np.cos(rad)

            sin_mean = sinv.rolling("60min", closed="right").mean()
            cos_mean = cosv.rolling("60min", closed="right").mean()

            # 平均矢量 -> 角度（0-360）
            ang = np.arctan2(sin_mean, cos_mean)
            deg = (np.rad2deg(ang) + 360.0) % 360.0
            out_parts.append(deg.to_frame(wind_dir_col))
        else:
            out_parts.append(rolling_obj[[wind_dir_col]].mean())

    if not out_parts:
        raise ValueError("weather 中没有可处理的字段（请检查列名）。")

    rolled = pd.concat(out_parts, axis=1)

    # 4) 只取整点小时：把每条“时刻 t 的滚动结果”对齐到 t 的整点版本
    #    例如 13:07 的滚动值会落到 13:00（我们只保留整点时刻的记录）
    rolled = rolled.reset_index()
    rolled["hour"] = rolled[dt_col].dt.floor("H")

    # 对同一小时可能有多条（因为原始是分钟级），这里只保留“该小时最后一条滚动结果”
    rolled = rolled.sort_values(dt_col).drop_duplicates(subset=["hour"], keep="last")
    rolled = rolled.drop(columns=[dt_col]).rename(columns={"hour": dt_col}).set_index(dt_col)

    # 5) 缺测窗口判定：如果窗口内观测条数 < min_obs，则整小时置为 NaN
    counts = (
        df.iloc[:, [0]]
        .rolling("60min", closed="right")
        .count()
        .reset_index()
    )
    counts["hour"] = counts[dt_col].dt.floor("H")
    counts = counts.sort_values(dt_col).drop_duplicates(subset=["hour"], keep="last")
    counts = counts.drop(columns=[dt_col]).set_index("hour").iloc[:, 0]  # Series

    rolled.loc[counts < min_obs, :] = np.nan

    # 6) 统一排序
    rolled = rolled.sort_index()

    return rolled


# ===== 用法示例 =====
# 假设你已经有 weather DataFrame
weather_hourly = hourly_inhour_rolling(
    weather,
    dt_col="datetime",
    wind_dir_col="wind_direction_deg",
    min_obs=1,
    strict_wind_dir=True,  # 建议论文用 True
)

weather_hourly.head(), weather_hourly.tail()

/tmp/ipykernel_44599/1916344644.py:65: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  rolled["hour"] = rolled[dt_col].dt.floor("H")
/tmp/ipykernel_44599/1916344644.py:78: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  counts["hour"] = counts[dt_col].dt.floor("H")


(                     temperature_C  pressure_hPa  relative_humidity_pct  \
 datetime                                                                  
 2020-01-01 00:00:00           17.9        1019.0                   82.0   
 2020-01-01 01:00:00           17.9        1018.9                   81.0   
 2020-01-01 02:00:00           17.7        1018.5                   82.0   
 2020-01-01 03:00:00           17.5        1018.3                   82.0   
 2020-01-01 04:00:00           17.2        1018.2                   85.0   
 
                      visibility_m  wind_speed_ms  precipitation_mm  \
 datetime                                                             
 2020-01-01 00:00:00       10746.0            0.8               0.0   
 2020-01-01 01:00:00       10957.0            1.3               0.0   
 2020-01-01 02:00:00       10035.0            0.7               0.0   
 2020-01-01 03:00:00       10267.0            0.9               0.0   
 2020-01-01 04:00:00       10398.0      

In [35]:
import pandas as pd

# 1️⃣ 确保是 datetime 类型
pm_detail["datetime"] = pd.to_datetime(pm_detail["datetime"], errors="coerce")

# 2️⃣ 提取 hh:mm:ss 并转为字符串
time_str_series = pm_detail["datetime"].dt.strftime("%H:%M:%S")

# 3️⃣ 转为 set
time_set = set(time_str_series.dropna())

print(time_set,len(time_set))

{'08:00:00', '23:00:00', '12:00:00', '03:00:00', '05:00:00', '17:00:00', '01:00:00', '14:00:00', '15:00:00', '04:00:00', '19:00:00', '20:00:00', '02:00:00', '06:00:00', '18:00:00', '00:00:00', '22:00:00', '21:00:00', '09:00:00', '11:00:00', '13:00:00', '10:00:00', '07:00:00', '16:00:00'} 24


In [ ]:
def _ensure_datetime_col(df: pd.DataFrame, col="datetime") -> pd.DataFrame:
    df = df.copy()
    if col not in df.columns and df.index.name == col:
        df = df.reset_index()
    df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

def _floor_to_hour(df: pd.DataFrame, col="datetime") -> pd.DataFrame:
    df = df.copy()
    df[col] = pd.to_datetime(df[col], errors="coerce").dt.floor("H")
    return df



In [40]:
# 1) 准备 weather_hourly（可能是索引）
weather_h = _ensure_datetime_col(weather_hourly, "datetime")
weather_h = _floor_to_hour(weather_h, "datetime")

weather_h = weather_h.sort_values("datetime").drop_duplicates("datetime", keep="last")
weather_h.head()

/tmp/ipykernel_44599/1197206374.py:12: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df[col] = pd.to_datetime(df[col], errors="coerce").dt.floor("H")


,datetime,temperature_C,pressure_hPa,relative_humidity_pct,visibility_m,wind_speed_ms,precipitation_mm,wind_direction_deg
0,2020-01-01 00:00:00,17.9,1019.0,82.0,10746.0,0.8,0.0,106.0
1,2020-01-01 01:00:00,17.9,1018.9,81.0,10957.0,1.3,0.0,111.0
2,2020-01-01 02:00:00,17.7,1018.5,82.0,10035.0,0.7,0.0,105.0
3,2020-01-01 03:00:00,17.5,1018.3,82.0,10267.0,0.9,0.0,113.0
4,2020-01-01 04:00:00,17.2,1018.2,85.0,10398.0,0.7,0.0,24.0


In [41]:
pm_d = _ensure_datetime_col(pm_detail, "datetime")
pm_d = _floor_to_hour(pm_d, "datetime")
pm_d.head()

/tmp/ipykernel_44599/1197206374.py:12: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df[col] = pd.to_datetime(df[col], errors="coerce").dt.floor("H")


,datetime,site_id,average_pm25_hour,district
0,2020-01-01 03:00:00,通心岭,48.0,福田区
1,2020-01-01 04:00:00,华侨城,50.0,南山区
2,2020-01-01 04:00:00,南海,39.0,南山区
3,2020-01-01 04:00:00,南澳,35.0,大鹏新区
4,2020-01-01 04:00:00,葵涌,35.0,大鹏新区


In [47]:
print(set(pm_detail['district']))

{'大鹏新区', '罗湖区', '南山区', '龙华区', '龙岗区', '福田区', '坪山区', '盐田区', '宝安区'}


In [42]:
rea = _ensure_datetime_col(reanalysis_by_district, "datetime")
rea = _floor_to_hour(rea, "datetime")
rea = rea.rename(columns={"district": "target_district"})
rea.head()

/tmp/ipykernel_44599/1197206374.py:12: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df[col] = pd.to_datetime(df[col], errors="coerce").dt.floor("H")


,datetime,u10,v10,t2m_c,d2m_c,sp_hpa,blh_m,wind_speed,wind_dir,rh,target_district
0,2020-01-01 00:00:00,-2.662069,-1.045527,15.878062,12.418854,1020.297550,387.338620,2.869726,67.817326,79.974575,其它
1,2020-01-01 00:00:00,-5.173950,-2.068314,16.481903,13.555573,1021.944200,526.619900,5.572045,68.210660,82.841760,大鹏区
2,2020-01-01 01:00:00,-3.147481,-1.151866,15.898214,12.456838,1020.544115,459.398925,3.356667,69.468003,80.089157,其它
3,2020-01-01 01:00:00,-5.691101,-2.125824,16.141052,13.327606,1022.165800,560.398900,6.075175,69.517580,83.412170,大鹏区
4,2020-01-01 02:00:00,-3.309372,-1.326365,17.272919,12.413930,1020.825500,540.437750,3.570108,67.815725,73.181199,其它


In [43]:
main = rea.merge(weather_h, on="datetime", how="left")
main = main.merge(pm_d, on='datetime', how="left")
main.head()

,datetime,u10,v10,t2m_c,d2m_c,sp_hpa,blh_m,wind_speed,wind_dir,rh,...,temperature_C,pressure_hPa,relative_humidity_pct,visibility_m,wind_speed_ms,precipitation_mm,wind_direction_deg,site_id,average_pm25_hour,district
0,2020-01-01 00:00:00,-2.662069,-1.045527,15.878062,12.418854,1020.297550,387.338620,2.869726,67.817326,79.974575,...,17.9,1019.0,82.0,10746.0,0.8,0.0,106.0,NaN,NaN,NaN
1,2020-01-01 00:00:00,-5.173950,-2.068314,16.481903,13.555573,1021.944200,526.619900,5.572045,68.210660,82.841760,...,17.9,1019.0,82.0,10746.0,0.8,0.0,106.0,NaN,NaN,NaN
2,2020-01-01 01:00:00,-3.147481,-1.151866,15.898214,12.456838,1020.544115,459.398925,3.356667,69.468003,80.089157,...,17.9,1018.9,81.0,10957.0,1.3,0.0,111.0,NaN,NaN,NaN
3,2020-01-01 01:00:00,-5.691101,-2.125824,16.141052,13.327606,1022.165800,560.398900,6.075175,69.517580,83.412170,...,17.9,1018.9,81.0,10957.0,1.3,0.0,111.0,NaN,NaN,NaN
4,2020-01-01 02:00:00,-3.309372,-1.326365,17.272919,12.413930,1020.825500,540.437750,3.570108,67.815725,73.181199,...,17.7,1018.5,82.0,10035.0,0.7,0.0,105.0,NaN,NaN,NaN


In [44]:
main.dropna(inplace=True)
main.head()

,datetime,u10,v10,t2m_c,d2m_c,sp_hpa,blh_m,wind_speed,wind_dir,rh,...,temperature_C,pressure_hPa,relative_humidity_pct,visibility_m,wind_speed_ms,precipitation_mm,wind_direction_deg,site_id,average_pm25_hour,district
6,2020-01-01 03:00:00,-3.286784,-1.577240,17.453390,12.299967,1020.512612,620.750867,3.652529,63.997594,71.819595,...,17.5,1018.3,82.0,10267.0,0.9,0.0,113.0,通心岭,48.0,福田区
7,2020-01-01 03:00:00,-5.665527,-2.575287,16.488220,12.817871,1022.125900,603.646700,6.223368,65.555630,78.917090,...,17.5,1018.3,82.0,10267.0,0.9,0.0,113.0,通心岭,48.0,福田区
8,2020-01-01 04:00:00,-2.780365,-1.761871,17.413544,12.243256,1019.632922,722.791260,3.304431,57.119415,71.759914,...,17.2,1018.2,85.0,10398.0,0.7,0.0,24.0,华侨城,50.0,南山区
9,2020-01-01 04:00:00,-2.780365,-1.761871,17.413544,12.243256,1019.632922,722.791260,3.304431,57.119415,71.759914,...,17.2,1018.2,85.0,10398.0,0.7,0.0,24.0,南海,39.0,南山区
10,2020-01-01 04:00:00,-2.780365,-1.761871,17.413544,12.243256,1019.632922,722.791260,3.304431,57.119415,71.759914,...,17.2,1018.2,85.0,10398.0,0.7,0.0,24.0,南澳,35.0,大鹏新区


In [52]:
# 1) main：提取天级时间戳（每天 00:00）
main = main.copy()
main["datetime"] = pd.to_datetime(main["datetime"], errors="coerce")
main["date00"] = main["datetime"].dt.floor("D")   # = 每天 00:00:00
# 2) air_quality：把 datetime 也处理成每天 00:00，用同名键 date00
air = air_quality.copy()
air["datetime"] = pd.to_datetime(air["datetime"], errors="coerce")
air["date00"] = air["datetime"].dt.floor("D")
# 4) 合并：推荐按 [target_district, date00]（天级目标 + 区）
# 注意：main 是小时级，会导致同一天多行；合并后仍会是小时级行数（这是预期）
merged = main.merge(air, on=["date00",'district'], how="left", suffixes=("", "_air"))

merged.head()

,datetime,u10,v10,t2m_c,d2m_c,sp_hpa,blh_m,wind_speed,wind_dir,rh,...,average_pm25_hour,district,date00,O3,PM25,CO,NO2,datetime_air,SO2,PM10
0,2020-01-01 03:00:00,-3.286784,-1.577240,17.453390,12.299967,1020.512612,620.750867,3.652529,63.997594,71.819595,...,48.0,福田区,2020-01-01,67.0,46.0,0.80,20.0,2020-01-01,6.0,72.0
1,2020-01-01 03:00:00,-5.665527,-2.575287,16.488220,12.817871,1022.125900,603.646700,6.223368,65.555630,78.917090,...,48.0,福田区,2020-01-01,67.0,46.0,0.80,20.0,2020-01-01,6.0,72.0
2,2020-01-01 04:00:00,-2.780365,-1.761871,17.413544,12.243256,1019.632922,722.791260,3.304431,57.119415,71.759914,...,50.0,南山区,2020-01-01,47.5,42.0,0.65,29.5,2020-01-01,4.5,75.5
3,2020-01-01 04:00:00,-2.780365,-1.761871,17.413544,12.243256,1019.632922,722.791260,3.304431,57.119415,71.759914,...,39.0,南山区,2020-01-01,47.5,42.0,0.65,29.5,2020-01-01,4.5,75.5
4,2020-01-01 04:00:00,-2.780365,-1.761871,17.413544,12.243256,1019.632922,722.791260,3.304431,57.119415,71.759914,...,35.0,大鹏新区,2020-01-01,68.5,34.0,0.75,11.5,2020-01-01,5.5,49.5


In [53]:
merged=merged[((merged['target_district']=='大鹏区') & (merged['district']=='大鹏新区')) | ((merged['target_district']!='大鹏区') & (merged['district']!='大鹏新区'))]

In [54]:
merged.head()

,datetime,u10,v10,t2m_c,d2m_c,sp_hpa,blh_m,wind_speed,wind_dir,rh,...,average_pm25_hour,district,date00,O3,PM25,CO,NO2,datetime_air,SO2,PM10
0,2020-01-01 03:00:00,-3.286784,-1.577240,17.453390,12.299967,1020.512612,620.750867,3.652529,63.997594,71.819595,...,48.0,福田区,2020-01-01,67.0,46.0,0.80,20.0,2020-01-01,6.0,72.0
2,2020-01-01 04:00:00,-2.780365,-1.761871,17.413544,12.243256,1019.632922,722.791260,3.304431,57.119415,71.759914,...,50.0,南山区,2020-01-01,47.5,42.0,0.65,29.5,2020-01-01,4.5,75.5
3,2020-01-01 04:00:00,-2.780365,-1.761871,17.413544,12.243256,1019.632922,722.791260,3.304431,57.119415,71.759914,...,39.0,南山区,2020-01-01,47.5,42.0,0.65,29.5,2020-01-01,4.5,75.5
6,2020-01-01 04:00:00,-2.780365,-1.761871,17.413544,12.243256,1019.632922,722.791260,3.304431,57.119415,71.759914,...,54.0,宝安区,2020-01-01,53.0,49.0,0.90,41.0,2020-01-01,6.0,88.0
7,2020-01-01 04:00:00,-2.780365,-1.761871,17.413544,12.243256,1019.632922,722.791260,3.304431,57.119415,71.759914,...,45.0,盐田区,2020-01-01,60.0,36.0,0.70,26.5,2020-01-01,6.5,61.0


In [55]:
merged.to_csv('tmp.csv', index=False)